# Práctica de clasificación de siluetas: conjunto de datos y carga

En esta primera sección conocerás el origen de los datos y los cargarás en arreglos separados para entrenamiento y prueba. Conserva esta partición durante toda la práctica. Las celdas siguientes de este notebook servirán de base para tus experimentos.

**Al terminar tendrás:** `X_train`, `y_train`, `X_test` y `y_test`, además de los identificadores de las siluetas originales para organizar más adelante la validación cruzada.


## 1. Descripción del conjunto de datos

**MPEG-7 Core Experiment CE-Shape-1, Part B** es una colección de siluetas de objetos utilizada para estudiar la descripción y comparación de formas. El conjunto original contiene **70 clases**, con **20 siluetas por clase**, para un total de **1400 siluetas**. Se puede consultar una imagen de ejemplo por clase y el conjunto original en la [página del grupo de Longin Jan Latecki](https://dabi.temple.edu/external/shape/MPEG7/dataset.html).

En esta práctica trabajaremos con **diez clases seleccionadas** del conjunto original. El archivo entregado para la práctica contiene **1000 imágenes**, distribuidas de manera equilibrada: **100 por clase**, de las cuales **70 están en entrenamiento y 30 en prueba**. Cada clase proviene de 20 siluetas originales distintas. La variable objetivo es la **etiqueta de clase**; el problema es de clasificación multiclase.

| Etiqueta en los archivos | Objeto o figura representada |
|---|---|
| `Bone` | Hueso |
| `bottle` | Botella |
| `brick` | Ladrillo o bloque |
| `butterfly` | Mariposa |
| `fork` | Tenedor |
| `Heart` | Corazón |
| `guitar` | Guitarra |
| `octopus` | Pulpo |
| `tree` | Árbol |
| `watch` | Reloj |



## 2. Carga de entrenamiento y prueba

Coloca el archivo **`MPEG7_10_clases_train_test.zip`** junto a este notebook. No necesitas descomprimirlo. Dentro del ZIP, las imágenes están organizadas en `entrenamiento/<clase>/` y `prueba/<clase>/`, y el archivo `etiquetas_y_particion.csv` contiene sus rutas y etiquetas.

Usaremos `numpy`, `pandas` y `Pillow`. Ejecuta las celdas en orden y examina las formas de los arreglos que se imprimen al final.


In [1]:
# Bibliotecas para rutas, lectura de ZIP, tablas e imágenes.
from pathlib import Path
from zipfile import ZipFile
from io import BytesIO

import numpy as np
import pandas as pd
from PIL import Image

# El ZIP debe estar en la misma carpeta que el notebook.
RUTA_ZIP = Path('MPEG7_10_clases_train_test.zip')
if not RUTA_ZIP.is_file():
    raise FileNotFoundError(
        f'No se encontró {RUTA_ZIP.name}. Colócalo junto al notebook.'
    )

# Establecemos un orden explícito para las clases.
CLASES = ['Bone', 'bottle', 'brick', 'butterfly', 'fork',
          'Heart', 'guitar', 'octopus', 'tree', 'watch']

# Leemos la tabla de metadatos directamente desde el archivo comprimido.
with ZipFile(RUTA_ZIP) as zip_datos:
    metadatos = pd.read_csv(zip_datos.open('etiquetas_y_particion.csv'))
    rutas_zip = set(zip_datos.namelist())

# Verificaciones básicas para detectar un ZIP equivocado o incompleto.
assert len(metadatos) == 1000, 'Se esperaban 1000 imágenes.'
assert set(metadatos['clase']) == set(CLASES), 'Las clases no coinciden.'
assert metadatos['archivo'].is_unique, 'Hay rutas duplicadas.'
assert set(metadatos['archivo']).issubset(rutas_zip), 'Faltan imágenes en el ZIP.'

print('Primeras filas de la tabla de metadatos:')
display(metadatos[['archivo', 'clase', 'particion', 'original_mpeg7']].head())


AssertionError: Faltan imágenes en el ZIP.

In [ ]:
def cargar_particion(zip_datos, tabla, nombre_particion):
    """Lee imágenes y etiquetas de una partición del ZIP.

    Parámetros
    ----------
    zip_datos : ZipFile
        Archivo comprimido abierto para lectura.
    tabla : pandas.DataFrame
        Metadatos con ruta, clase, partición y silueta original.
    nombre_particion : str
        'entrenamiento' o 'prueba'.

    Retorna
    -------
    X : ndarray de forma (n, 64, 64)
        Imágenes en escala de grises con valores 0 y 255.
    y : ndarray de forma (n,)
        Etiquetas de clase en texto.
    grupos : ndarray de forma (n,)
        Identificadores de la silueta original.
    """
    # Filtramos sin mezclar las particiones. Ordenar por ruta permite
    # reproducir siempre el mismo orden de imágenes y etiquetas.
    seleccion = (tabla.loc[tabla['particion'] == nombre_particion]
                 .sort_values('archivo').reset_index(drop=True))

    imagenes = []
    for ruta in seleccion['archivo']:
        # BytesIO convierte los bytes internos del ZIP en un archivo legible
        # por Pillow. El arreglo conserva los píxeles de la imagen.
        contenido = zip_datos.read(ruta)
        with Image.open(BytesIO(contenido)) as imagen:
            arreglo = np.asarray(imagen.convert('L'), dtype=np.uint8)
        imagenes.append(arreglo)

    X = np.stack(imagenes, axis=0)
    y = seleccion['clase'].to_numpy()
    grupos = seleccion['original_mpeg7'].to_numpy()
    return X, y, grupos


# Cargamos las dos particiones desde las carpetas indicadas en el CSV.
with ZipFile(RUTA_ZIP) as zip_datos:
    X_train, y_train, grupos_train = cargar_particion(
        zip_datos, metadatos, 'entrenamiento')
    X_test, y_test, grupos_test = cargar_particion(
        zip_datos, metadatos, 'prueba')

print('X_train:', X_train.shape, X_train.dtype)
print('y_train:', y_train.shape)
print('X_test: ', X_test.shape, X_test.dtype)
print('y_test: ', y_test.shape)


In [ ]:
# Confirmamos dimensiones, valores y distribución de clases.
assert X_train.shape == (700, 64, 64)
assert X_test.shape == (300, 64, 64)
assert np.isin(np.unique(X_train), [0, 255]).all()
assert np.isin(np.unique(X_test), [0, 255]).all()

conteos = pd.DataFrame({
    'entrenamiento': pd.Series(y_train).value_counts(),
    'prueba': pd.Series(y_test).value_counts()
}).reindex(CLASES)
display(conteos)

assert (conteos['entrenamiento'] == 70).all()
assert (conteos['prueba'] == 30).all()

# Los identificadores de origen se conservan para una validación cruzada
# posterior. Ninguna silueta original aparece en ambos conjuntos.
assert not (set(grupos_train) & set(grupos_test))
print('Siluetas originales en entrenamiento:', len(set(grupos_train)))
print('Siluetas originales en prueba:', len(set(grupos_test)))


## 3. Visualización: dos imágenes por clase

Mostraremos **dos imágenes de entrenamiento de cada clase**, procedentes de **siluetas originales distintas**. Observa qué rasgos se conservan dentro de cada clase y cuáles cambian. Esta selección es solo ilustrativa; no representa necesariamente toda la variación de la clase.


In [ ]:
# Utilizamos únicamente imágenes de entrenamiento para esta inspección.
# Elegimos una imagen de cada una de dos siluetas originales diferentes.
import matplotlib.pyplot as plt

fig, axes = plt.subplots(len(CLASES), 2, figsize=(5, 2.0 * len(CLASES)))

with ZipFile(RUTA_ZIP) as zip_datos:
    for fila, clase in enumerate(CLASES):
        # Ordenar y eliminar originales repetidos hace reproducible la selección.
        ejemplos = (metadatos.loc[
            (metadatos['particion'] == 'entrenamiento') &
            (metadatos['clase'] == clase)
        ].sort_values(['original_mpeg7', 'archivo'])
         .drop_duplicates(subset='original_mpeg7')
         .head(2))
        assert len(ejemplos) == 2, f'Faltan originales para {clase}.'

        for columna, (_, registro) in enumerate(ejemplos.iterrows()):
            # Leemos la ruta registrada en el CSV directamente desde el ZIP.
            contenido = zip_datos.read(registro['archivo'])
            with Image.open(BytesIO(contenido)) as imagen:
                arreglo = np.asarray(imagen.convert('L'))
            axes[fila, columna].imshow(arreglo, cmap='gray', vmin=0, vmax=255)
            axes[fila, columna].set_title(
                f'{clase} · ejemplo {columna + 1}', fontsize=10)
            axes[fila, columna].axis('off')

fig.suptitle('Dos siluetas originales distintas por clase', fontsize=14, y=1.005)
plt.tight_layout()
plt.show()


## 4. Caracterización de las imágenes

Un clasificador aprende a partir de **vectores de características**. La caracterización decide qué información de la imagen entra al modelo: una representación puede conservar muchos detalles, pero también registrar variaciones que no ayudan a distinguir clases; otra puede ser compacta y estable, pero perder detalles útiles. Por eso, la elección de características influye directamente en la separación entre clases y en la generalización a imágenes nuevas.

En esta práctica construiremos dos representaciones de cada silueta de $64\times64$ píxeles:

| Representación | Características por imagen | Idea |
|---|---:|---|
| Valores de píxeles | $64\times64=4096$ | Aplanar la imagen: cada posición aporta una característica. Conserva el detalle espacial y depende de dónde aparece cada parte de la figura. |
| Momentos invariantes de Hu | $7$ | Resumir cómo se distribuye la masa de la silueta mediante combinaciones de momentos geométricos normalizados. |

Los **siete momentos de Hu** describen propiedades globales de la forma. En la formulación matemática ideal son invariantes frente a traslación, escala y rotación; en imágenes digitales la invariancia es aproximada por la discretización. Esto los hace útiles cuando el objeto puede cambiar de posición, tamaño u orientación. Su compacidad también tiene un costo: dos contornos distintos pueden producir descripciones parecidas y ciertos detalles locales se pierden. El séptimo momento puede cambiar de signo ante una reflexión.

Usaremos los mismos ejemplos y etiquetas para ambas representaciones. **No modificaremos `X_train` ni `X_test`:** crearemos arreglos nuevos, cuyos nombres indican la representación que contienen.


In [ ]:
# Cada imagen tiene 64 × 64 posiciones: la convertimos en un vector de 4096.
# La conversión a float permite trabajar con los valores en modelos numéricos.
# Dividir entre 255 cambia 0/255 por 0/1, sin alterar la información binaria.
X_train_pixeles = X_train.reshape(len(X_train), -1).astype(np.float32) / 255.0
X_test_pixeles = X_test.reshape(len(X_test), -1).astype(np.float32) / 255.0

# Comprobamos que cada fila corresponde a una imagen y cada columna a un píxel.
assert X_train_pixeles.shape == (700, 4096)
assert X_test_pixeles.shape == (300, 4096)
print('Píxeles, entrenamiento:', X_train_pixeles.shape)
print('Píxeles, prueba:       ', X_test_pixeles.shape)
print('Valores posibles:', np.unique(X_train_pixeles))


## 5. Momentos invariantes de Hu

Primero calcularemos los siete momentos para cada máscara binaria. Sus magnitudes pueden diferir en muchos órdenes, por lo que aplicaremos una transformación logarítmica que conserva el signo. Finalmente, estandarizaremos cada característica utilizando **solo los datos de entrenamiento**; la misma transformación se aplicará después a prueba. Esto evita que la información del conjunto de prueba intervenga en la preparación del modelo.

La transformación logarítmica es una decisión numérica para comparar los momentos en una escala manejable. No cambia el número de características: cada imagen sigue representada por siete valores.

Si falta OpenCV en tu entorno, instala el paquete `opencv-python` antes de ejecutar la siguiente celda.


In [ ]:
import cv2
from sklearn.preprocessing import StandardScaler


def obtener_hu(imagen):
    """Devuelve los siete momentos de Hu de una máscara binaria.

    Parámetros
    ----------
    imagen : ndarray de forma (64, 64)
        Silueta con fondo 0 y figura 255.
    """
    # binaryImage=True interpreta todos los píxeles distintos de cero como 1.
    momentos = cv2.moments(imagen, binaryImage=True)
    return cv2.HuMoments(momentos).ravel()


def transformar_hu(valores):
    """Conserva el signo y comprime el rango de magnitudes de Hu."""
    # El piso evita calcular log10(0) en valores nulos o muy pequeños.
    magnitudes = np.maximum(np.abs(valores), 1e-30)
    return -np.sign(valores) * np.log10(magnitudes)


# Calculamos Hu individualmente, respetando el orden de las imágenes cargadas.
X_train_hu_log = np.vstack([transformar_hu(obtener_hu(imagen))
                             for imagen in X_train])
X_test_hu_log = np.vstack([transformar_hu(obtener_hu(imagen))
                            for imagen in X_test])

# Cada columna se centra y escala con estadísticas de ENTRENAMIENTO.
escalador_hu = StandardScaler()
X_train_hu = escalador_hu.fit_transform(X_train_hu_log)
X_test_hu = escalador_hu.transform(X_test_hu_log)

assert X_train_hu.shape == (700, 7)
assert X_test_hu.shape == (300, 7)
assert np.isfinite(X_train_hu).all() and np.isfinite(X_test_hu).all()
print('Hu, entrenamiento:', X_train_hu.shape)
print('Hu, prueba:       ', X_test_hu.shape)


## 6. Visualización exploratoria con t-SNE

Las representaciones tienen **4096** y **7** dimensiones, respectivamente, por lo que no podemos observarlas directamente en un plano. **t-SNE** construye una proyección a dos dimensiones que intenta conservar relaciones de vecindad: puntos cercanos en la gráfica suelen corresponder a ejemplos similares según la representación utilizada. Al colorear los puntos y escribir el nombre de cada clase cerca de la mediana de sus ejemplos, podemos explorar posibles agrupamientos y confusiones entre clases.

Calcularemos las proyecciones **solo con entrenamiento**. Las etiquetas se usan únicamente para colorear después de la proyección: t-SNE no las recibe al ajustarse. Se fijará la semilla para reproducir cada gráfica.

**Interpretación:** busca clases que se mezclan o forman grupos locales. Las coordenadas, la orientación y las distancias entre grupos lejanos no tienen una interpretación directa, y **los ejes de las dos gráficas no son comparables entre sí**. Una separación visual tampoco demuestra que un clasificador generalice.


In [ ]:
from sklearn.manifold import TSNE

# Fijamos los mismos parámetros para ambas proyecciones. Cada ajuste de
# t-SNE se hace por separado y recibe solamente las características.
config_tsne = dict(n_components=2, perplexity=30, init='pca',
                   learning_rate='auto', random_state=42)

# Utilizamos únicamente entrenamiento para explorar las representaciones.
proyeccion_pixeles = TSNE(**config_tsne).fit_transform(X_train_pixeles)
proyeccion_hu = TSNE(**config_tsne).fit_transform(X_train_hu)

# Los colores representan etiquetas conocidas; no se emplearon al ajustar t-SNE.
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
colores = plt.get_cmap('tab10')

for ax, (titulo, coordenadas) in zip(
    axes,
    [('Píxeles (4096 características)', proyeccion_pixeles),
     ('Hu (7 características)', proyeccion_hu)]
):
    for indice, clase in enumerate(CLASES):
        seleccion = (y_train == clase)
        puntos = coordenadas[seleccion]
        ax.scatter(puntos[:, 0], puntos[:, 1],
                   s=18, alpha=0.7, color=colores(indice), label=clase)

        # Escribimos la clase cerca de la mediana de sus puntos.
        # En grupos superpuestos la etiqueta señala una zona aproximada,
        # no una frontera de clasificación.
        mediana = np.median(puntos, axis=0)
        # Anclamos el rótulo en un ejemplo real cercano a la mediana;
        # así no queda en un espacio vacío si la clase tiene varios grupos.
        centro = puntos[np.argmin(np.linalg.norm(puntos - mediana, axis=1))]
        desplazamiento = {'tree': (5, 14), 'brick': (5, -12)}.get(
            clase, (5, 5))
        ax.annotate(clase, xy=(centro[0], centro[1]),
                    xytext=desplazamiento, textcoords='offset points',
                    fontsize=8, weight='bold', color=colores(indice),
                    bbox=dict(facecolor='white', edgecolor='none',
                              alpha=0.82, pad=1.5))
    ax.set_title(titulo)
    ax.set_xlabel('Componente t-SNE 1')
    ax.set_ylabel('Componente t-SNE 2')
    ax.grid(alpha=0.15)

# Una sola leyenda mantiene libre el espacio de cada proyección.
handles, etiquetas = axes[1].get_legend_handles_labels()
fig.legend(handles, etiquetas, loc='lower center', ncol=5,
           bbox_to_anchor=(0.5, -0.06))
fig.suptitle('Distribución exploratoria del conjunto de entrenamiento', y=1.01)
plt.tight_layout()
plt.show()


### Preguntas breves

1. ¿Qué clases parecen formar vecindarios más definidos en cada gráfica?
2. ¿Qué clases aparecen mezcladas? Relaciona tu observación con las siluetas vistas en la sección 3.



## 7. Entrenamiento y evaluación de clasificadores

**Actividad.** Configura, entrena y prueba **cada algoritmo con ambas representaciones** (píxeles y Hu):

1. **Mínima distancia** (centroide de cada clase).
2. **KNN**.
3. **Perceptrón**.
4. **Perceptrón multicapa**: una capa oculta de **100 neuronas**.
5. **Máquina de soporte vectorial (SVM)**.
6. **Árbol de decisión**.
7. **Bayes ingenuo**: usa la variante **gaussiana**.
8. **Regresión logística**.

Utiliza **los hiperparámetros predeterminados de cada algoritmo**, salvo el perceptrón multicapa, donde debes establecer explícitamente la capa oculta de 100 neuronas. Para poder reproducir la ejecución de este último puedes fijar su semilla; si necesitas aumentar su número máximo de iteraciones, informa el valor empleado y por qué. No selecciones hiperparámetros mirando los resultados de prueba.

**Multiclase.** Aunque suelen introducirse primero en problemas de dos clases, el perceptrón, la máquina de soporte vectorial y la regresión logística también se aplican aquí: el perceptrón de `scikit-learn` construye una decisión para cada clase frente al resto; la regresión logística con el solucionador predeterminado maneja las diez clases mediante un modelo multinomial. Investiga qué significan estas dos estrategias y comprueba que cada modelo tenga diez etiquetas en `classes_` después del ajuste. ¿Cómo se implementa la máquina de soporte vectorial para problemas multiclase?

**Preparación por representación.** Construye un `Pipeline` independiente por algoritmo y representación. Los píxeles ya están codificados como 0/1; los siete valores de Hu transformados logarítmicamente requieren `StandardScaler`. Ajusta el escalador **solo con entrenamiento** mediante el pipeline; por eso usa `X_train_hu_log` y `X_test_hu_log` en esta actividad, no los arreglos `X_train_hu` y `X_test_hu` previamente escalados. Para los píxeles puedes utilizar un pipeline que contenga únicamente el clasificador. El conjunto de prueba solo se pasa a `predict` y a las métricas.

**Métricas.** Calcula en **entrenamiento y prueba**:

- **Exactitud (*accuracy*):** proporción de imágenes correctamente clasificadas.
- **F1 macro:** calcula primero, para cada clase, precisión $P_c$ y exhaustividad (*recall*) $R_c$; luego $F1_c=2P_cR_c/(P_c+R_c)$. Finalmente promedia los diez valores $F1_c$ **con el mismo peso por clase**. Investiga cómo se tratan los denominadores nulos y explica qué significa obtener un F1 macro bajo aun cuando la exactitud sea mayor.

Los 1000 archivos proceden de 200 siluetas originales. La partición entregada mantiene separados los orígenes entre entrenamiento y prueba. **Una sola partición produce un valor de prueba por configuración:** no reportes media y desviación estándar entre particiones que no se realizaron.


In [ ]:
# COMPLETA esta celda con los ocho estimadores solicitados.
# Consulta sus constructores en sklearn.neighbors, sklearn.linear_model,
# sklearn.neural_network, sklearn.svm, sklearn.tree y sklearn.naive_bayes.
# Usa valores por defecto excepto para la configuración indicada del MLP.

from sklearn.base import clone
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score

NOMBRES_MODELOS = [
    'Mínima distancia', 'KNN', 'Perceptrón', 'Perceptrón multicapa',
    'SVM', 'Árbol de decisión', 'Bayes ingenuo', 'Regresión logística'
]

# Sustituye cada None por una instancia del clasificador correspondiente.
# Ejemplo de la estructura: 'Nombre': ConstructorDelClasificador()
modelos = {nombre: None for nombre in NOMBRES_MODELOS}

# Pistas: para mínima distancia busca NearestCentroid;
# para Bayes ingenuo utiliza GaussianNB.


In [ ]:
# Esta celda se ejecuta después de sustituir todos los None de `modelos`.
# En ambos casos se clona el estimador, para entrenar un modelo nuevo por
# representación sin reutilizar un modelo ya ajustado.
import pandas as pd

resultados = []
predicciones_guardadas = {}
pendientes = [nombre for nombre, estimador in modelos.items()
              if estimador is None]

if pendientes:
    print('Completa primero los estimadores:', ', '.join(pendientes))
else:
    for nombre, estimador in modelos.items():
        for representacion, X_ent, X_prueba in [
            ('Píxeles', X_train_pixeles, X_test_pixeles),
            ('Hu', X_train_hu_log, X_test_hu_log)
        ]:
            # Para Hu, la estandarización aprende exclusivamente de X_ent.
            # Para píxeles, el pipeline contiene solo el clasificador.
            if representacion == 'Hu':
                pipeline = make_pipeline(StandardScaler(), clone(estimador))
            else:
                pipeline = make_pipeline(clone(estimador))

            pipeline.fit(X_ent, y_train)

            # Predicciones y métricas para ambos conjuntos.
            for conjunto, X, y in [
                ('Entrenamiento', X_ent, y_train),
                ('Prueba', X_prueba, y_test)
            ]:
                predicciones = pipeline.predict(X)
                # Conservamos etiquetas reales y predicciones para las
                # matrices de confusión de la siguiente sección.
                predicciones_guardadas[(nombre, representacion, conjunto)] = (
                    y.copy(), predicciones.copy())
                resultados.append({
                    'Algoritmo': nombre,
                    'Representación': representacion,
                    'Conjunto': conjunto,
                    'Exactitud': accuracy_score(y, predicciones),
                    'F1 macro': f1_score(y, predicciones, average='macro')
                })

    resultados = pd.DataFrame(resultados)
    assert len(resultados) == 8 * 2 * 2
    display(resultados.head())


## 8. Tablas de resultados y matrices de confusión

Presenta **una tabla por algoritmo**. Cada tabla debe contener las cuatro combinaciones de conjunto y representación, con los valores de exactitud y F1 macro. La siguiente celda construye las ocho tablas a partir de `resultados`, una vez que hayas configurado y ejecutado los modelos en la sección 7. Una raya indica que todavía falta realizar esa evaluación. Después, construye **cuatro matrices de confusión por algoritmo**: entrenamiento y prueba, cada uno con píxeles y con Hu. Usa el mismo orden de clases en todas las matrices, con las clases reales en las filas y las predichas en las columnas. La diagonal muestra aciertos; las celdas fuera de la diagonal muestran confusiones.


In [ ]:
# Una tabla independiente por algoritmo. Si aún no se han entrenado
# modelos, se muestran las 8 tablas con espacios para completar.

columnas = ['Representación', 'Conjunto', 'Exactitud', 'F1 macro']
combinaciones = [(r, c) for r in ['Píxeles', 'Hu']
                 for c in ['Entrenamiento', 'Prueba']]

for nombre in NOMBRES_MODELOS:
    print(f'\n{nombre}')
    if isinstance(resultados, pd.DataFrame) and not resultados.empty:
        tabla_modelo = resultados.loc[
            resultados['Algoritmo'] == nombre, columnas
        ].copy()
        tabla_modelo['Exactitud'] = tabla_modelo['Exactitud'].map(
            lambda x: f'{x:.4f}')
        tabla_modelo['F1 macro'] = tabla_modelo['F1 macro'].map(
            lambda x: f'{x:.4f}')
    else:
        tabla_modelo = pd.DataFrame([
            {'Representación': r, 'Conjunto': c,
             'Exactitud': '—', 'F1 macro': '—'}
            for r, c in combinaciones
        ])
    display(tabla_modelo.reset_index(drop=True))


### Matrices de confusión

Ejecuta la celda siguiente después del entrenamiento. Generará **una figura por algoritmo** con las cuatro combinaciones de representación y conjunto. Cada matriz contiene conteos absolutos, no porcentajes. Comprueba que las filas sean clases reales y las columnas sean clases predichas.


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
import matplotlib.pyplot as plt

if not predicciones_guardadas:
    print('Configura y entrena primero los ocho modelos en la sección 7.')
else:
    for nombre in NOMBRES_MODELOS:
        # Filas: representación. Columnas: conjunto.
        fig, axes = plt.subplots(2, 2, figsize=(17, 14))
        for fila, representacion in enumerate(['Píxeles', 'Hu']):
            for columna, conjunto in enumerate(['Entrenamiento', 'Prueba']):
                y_real, y_pred = predicciones_guardadas[
                    (nombre, representacion, conjunto)]

                # El orden común CLASES hace comparables las cuatro matrices.
                matriz = confusion_matrix(y_real, y_pred, labels=CLASES)
                visual = ConfusionMatrixDisplay(
                    confusion_matrix=matriz, display_labels=CLASES)
                visual.plot(ax=axes[fila, columna], cmap='Blues',
                            values_format='d', colorbar=False)
                axes[fila, columna].set_title(
                    f'{representacion} · {conjunto}', fontsize=12)
                axes[fila, columna].tick_params(axis='x', labelrotation=45)

        fig.suptitle(f'Matrices de confusión: {nombre}', fontsize=16)
        plt.tight_layout()
        plt.show()
        plt.close(fig)


### Análisis solicitado

1. Para cada algoritmo, compara entrenamiento y prueba usando exactitud y F1 macro. ¿Hay diferencias grandes que sugieran ajuste excesivo o un modelo con dificultades para aprender?
2. Compara píxeles y Hu **dentro del mismo algoritmo**. Señala casos en que las dos métricas conducen a conclusiones distintas.
3. Examina las matrices de confusión de **prueba**: identifica los pares de clases que se confunden más y relaciona esos errores con sus siluetas. Distingue los errores en cada representación.
4. Contrasta las matrices de entrenamiento y prueba. ¿Qué clases se reconocen bien en entrenamiento pero empeoran en prueba?
5. Explica cómo el F1 macro refleja el desempeño de **cada clase con el mismo peso**. ¿Por qué puede diferir de la exactitud aunque el conjunto esté equilibrado?
6. Anota cualquier aviso de convergencia y cómo limita la interpretación. No cambies silenciosamente los hiperparámetros después de observar la prueba.

La evaluación corresponde a la partición proporcionada. Una práctica posterior podrá estudiar estabilidad mediante validación cruzada agrupada por silueta original.
